# imports

In [10]:
%reload_ext autoreload
%autoreload 2

import torch
import sys
import os
from torchmetrics.functional.pairwise import pairwise_cosine_similarity

# local imports
sys.path.insert(
    0,
    os.path.join(
        (os.path.dirname(os.path.abspath(''))),
    )
)
from evaluation.bright.retrievers import (
    get_scores,
    calculate_retrieval_metrics
)
sys.path.pop(0)


PKLS_PATH = os.path.join(os.path.dirname(os.path.abspath('')), 'evaluation', 'bright')

# functions

In [14]:
def compute_results(query_emb, doc_emb, documents, ground_truth, query_ids, doc_ids, excluded_ids):
    scores = pairwise_cosine_similarity(torch.from_numpy(query_emb), torch.from_numpy(doc_emb))
    scores = scores.tolist()
    assert len(scores) == len(query_ids), f"{len(scores)}, {len(query_ids)}"
    assert len(scores[0]) == len(documents), f"{len(scores[0])}, {len(documents)}"
    final_scores = get_scores(query_ids=query_ids,doc_ids=doc_ids,scores=scores,excluded_ids=excluded_ids)
    results = calculate_retrieval_metrics(results=final_scores, qrels=ground_truth)
    # print(results)
    return results


# reproduce numbers

In [9]:
doc_ids, query_ids, excluded_ids = torch.load(os.path.join(PKLS_PATH, 'ids.pkl'))
documents, doc_emb = torch.load(os.path.join(PKLS_PATH, 'base_doc_emb.pkl'))
queries_base, query_emb_base = torch.load(os.path.join(PKLS_PATH, 'base_emb.pkl'))
queries_reason, query_emb_reason = torch.load(os.path.join(PKLS_PATH, 'reason_emb.pkl'))
ground_truth = torch.load(os.path.join(PKLS_PATH, 'ground_truth.pkl'))

In [15]:
results_base =compute_results(
    query_emb=query_emb_base,
    doc_emb=doc_emb,
    documents=documents,
    ground_truth=ground_truth,
    query_ids=query_ids,
    doc_ids=doc_ids,
    excluded_ids=excluded_ids
)

{'NDCG@1': 0.19737, 'NDCG@5': 0.25338, 'NDCG@10': 0.27224, 'NDCG@25': 0.29811, 'NDCG@50': 0.31211, 'NDCG@100': 0.33633, 'MAP@1': 0.10746, 'MAP@5': 0.21089, 'MAP@10': 0.22015, 'MAP@25': 0.22732, 'MAP@50': 0.22926, 'MAP@100': 0.23172, 'Recall@1': 0.10746, 'Recall@5': 0.30279, 'Recall@10': 0.35542, 'Recall@25': 0.45849, 'Recall@50': 0.5177, 'Recall@100': 0.6391, 'P@1': 0.19737, 'P@5': 0.11579, 'P@10': 0.06711, 'P@25': 0.03211, 'P@50': 0.01868, 'P@100': 0.01197, 'MRR': 0.29551, 'Oracle NDCG@1': 0.77632, 'Oracle NDCG@5': 0.67497, 'Oracle NDCG@10': 0.6728, 'Oracle NDCG@25': 0.67467, 'Oracle NDCG@50': 0.67467, 'Oracle NDCG@100': 0.67909}


In [16]:
results_reason =compute_results(
    query_emb=query_emb_reason,
    doc_emb=doc_emb,
    documents=documents,
    ground_truth=ground_truth,
    query_ids=query_ids,
    doc_ids=doc_ids,
    excluded_ids=excluded_ids
)

{'NDCG@1': 0.31579, 'NDCG@5': 0.34635, 'NDCG@10': 0.36869, 'NDCG@25': 0.39197, 'NDCG@50': 0.41933, 'NDCG@100': 0.42691, 'MAP@1': 0.18061, 'MAP@5': 0.30298, 'MAP@10': 0.31194, 'MAP@25': 0.31989, 'MAP@50': 0.32471, 'MAP@100': 0.3256, 'Recall@1': 0.18061, 'Recall@5': 0.38941, 'Recall@10': 0.46178, 'Recall@25': 0.54151, 'Recall@50': 0.65962, 'Recall@100': 0.6969, 'P@1': 0.31579, 'P@5': 0.15, 'P@10': 0.08553, 'P@25': 0.04053, 'P@50': 0.025, 'P@100': 0.01329, 'MRR': 0.39604, 'Oracle NDCG@1': 0.82895, 'Oracle NDCG@5': 0.73188, 'Oracle NDCG@10': 0.72938, 'Oracle NDCG@25': 0.73255, 'Oracle NDCG@50': 0.73485, 'Oracle NDCG@100': 0.73485}


# learn projections